In [ ]:
# ============================================================
# CCRI Hazard Statistics Processing in GEE.
# Phase 1: calculate exposure counts and hazard statistics.Unified code for all admin 2 units.
# 1. Main hazard × country processing via `process_hazard_for_country()`
# 2. Detect and fill missing ADM2 units via tiled processing with local boundary caching

# Administrative Level: ADM2.
# Author: Angelly Pugliese, Ph.D.
# Date: April 2026
# ============================================================

In [ ]:
# ============================================================
# Configuration
# ============================================================
OUTPUT_FOLDER = "output_CO_updated"

# Optional: filter to specific countries (ISO3 codes) or None for all
# e.g. ["GRL"] or ["ABW","AGO","AIA", ...]
PROCESS_SPECIFIC_COUNTRIES = None

# Optional: filter to specific hazard codes or None for all
# e.g. ["volcano_100km_buff"]
PROCESS_SPECIFIC_HAZARDS = None

# Whether to run Phase 2 (detect and fill missing ADM2 units)
RUN_MISSING_DATA_FILL = True

# Path to local boundary files for geometry caching (used in Phase 2)
BOUNDS_FOLDER = "bounds"

In [ ]:
# ============================================================
# Imports
# ============================================================
import pandas as pd
import ee
import os
from pathlib import Path
from GEE_functions import GEEUtils, Hazard
from indicator_functions import IndicatorUtils

pd.set_option('display.max_columns', 500)

In [ ]:
# ============================================================
# Authenticate and initialize Google Earth Engine
# ============================================================
ee.Authenticate()
ee.Initialize(project="unicef-ccri")
unicef_data_source_path = "projects/unicef-ccri/assets"
utils = GEEUtils(ee)
indicator_utils = IndicatorUtils()

In [ ]:
# ============================================================
# Load hazard metadata
# ============================================================
hazard_info = pd.read_excel("hazard_info.xlsx")
hazard_list = [
    Hazard.from_dict(row, asset_prefix=unicef_data_source_path)
    for row in hazard_info.to_dict(orient="records")
]

for index, hazard in enumerate(hazard_list):
    print(f"{index}. {hazard.name}, {hazard.code}, {hazard.asset}")

In [ ]:
# ============================================================
# Get countries to process (with optional filtering)
# ============================================================
countries_to_process = utils.get_countries_to_process(PROCESS_SPECIFIC_COUNTRIES)
print(f"Countries to process ({len(countries_to_process)}): {countries_to_process}")

In [ ]:
# ============================================================
# Optional: filter out already-processed countries per hazard
# Scans existing partial files and removes countries that are done.
# ============================================================
def get_filtered_countries(hazard_code, countries_to_process, output_folder):
    """Return countries not yet processed for a given hazard."""
    partials_dir = Path(output_folder) / "hazards" / hazard_code / "partials"
    if not partials_dir.exists():
        return countries_to_process

    processed = set()
    for f in partials_dir.iterdir():
        if f.is_file() and f.suffix == ".csv":
            processed.add(f.name[:6])  # e.g. "CAN_V1"

    countries_no_pop = set(utils.countries_no_population)
    filtered = [c for c in countries_to_process if c not in processed and c not in countries_no_pop]
    return filtered

## Phase 1 — Main Hazard Processing

Loop over each hazard × country. Uses `process_hazard_for_country()` which internally handles
admin1/admin2 splitting for large countries (CAN, RUS, USA, etc.) and applies the standard
`reduceRegions` approach.

In [ ]:
# ============================================================
# Phase 1: Process hazards × countries
# ============================================================
for hazard in hazard_list:
    if PROCESS_SPECIFIC_HAZARDS and hazard.code not in PROCESS_SPECIFIC_HAZARDS:
        continue

    now_haz = pd.Timestamp.now()
    print(f"\n{'='*60}")
    print(f"Processing hazard: {hazard.name} ({hazard.code})")
    print(f"{'='*60}")

    hazard_images = utils.define_hazard(hazard)
    filtered_list = get_filtered_countries(hazard.code, countries_to_process, OUTPUT_FOLDER)
    print(f"Countries to process: {len(filtered_list)}")

    for country_ucode in filtered_list:
        now_cou = pd.Timestamp.now()
        try:
            results = utils.process_hazard_for_country(hazard_images, country_ucode, hazard.asset)
            elapsed = pd.Timestamp.now() - now_cou
            print(f"  {country_ucode}: {len(results)} rows in {elapsed}")
            utils.save_country_results(results, hazard.code, country_ucode, OUTPUT_FOLDER)
        except Exception as e:
            print(f"  Error for {country_ucode}: {e}")

    print(f"Completed {hazard.name} in {pd.Timestamp.now() - now_haz}")

## Phase 2 — Detect and Fill Missing ADM2 Units

After Phase 1 completes, scan the output for any ADM2 units that are missing from
the partial CSVs. These typically occur in very large countries (RUS, CAN) where the
standard `reduceRegions` approach times out for individual admin2 units.

Missing units are processed individually using **tiled statistics** with locally-cached
geometries from the `bounds/` folder.

In [ ]:
# ============================================================
# Phase 2, Step 1: Detect missing ADM2 units
# ============================================================
if RUN_MISSING_DATA_FILL:
    # Get the full reference list of ADM2 ucodes from the GEE asset
    ADMIN2_ASSET_FC = ee.FeatureCollection(
        f'{unicef_data_source_path}/{utils.ADMIN2_ASSET}'
    )
    all_adm2_ucodes = ADMIN2_ASSET_FC.aggregate_array("adm2_ucode").getInfo()
    print(f"Total ADM2 units in GEE asset: {len(all_adm2_ucodes)}")

    # Find missing ADM2 units per hazard
    errors = utils.find_missing_adm2(OUTPUT_FOLDER, all_adm2_ucodes)

    # Print summary
    total_missing = 0
    for haz_code, file_errors in errors.items():
        missing_count = sum(len(m) for _, m in file_errors)
        if missing_count > 0:
            total_missing += missing_count
            print(f"  {haz_code}: {missing_count} missing ADM2 units across {len(file_errors)} files")
    print(f"\nTotal missing ADM2 units across all hazards: {total_missing}")
else:
    print("Skipping Phase 2 (RUN_MISSING_DATA_FILL = False)")

In [ ]:
# ============================================================
# Phase 2, Step 2: Load local boundaries for geometry caching
# ============================================================
if RUN_MISSING_DATA_FILL and total_missing > 0:
    geojson_cache, properties_cache = GEEUtils.load_local_boundaries(BOUNDS_FOLDER)
else:
    geojson_cache, properties_cache = {}, {}

In [ ]:
# ============================================================
# Phase 2, Step 3: Fill missing ADM2 units using tiled processing
# ============================================================
if RUN_MISSING_DATA_FILL and total_missing > 0:
    for hazard_obj in hazard_list:
        if PROCESS_SPECIFIC_HAZARDS and hazard_obj.code not in PROCESS_SPECIFIC_HAZARDS:
            continue
        if hazard_obj.code not in errors:
            continue

        file_errors = errors[hazard_obj.code]
        hazard_missing = sum(len(m) for _, m in file_errors)
        if hazard_missing == 0:
            continue

        now_haz = pd.Timestamp.now()
        print(f"\n{'='*60}")
        print(f"Filling {hazard_missing} missing units for: {hazard_obj.name}")
        print(f"{'='*60}")

        hazard_images = utils.define_hazard(hazard_obj)

        for filename, missing_list in file_errors:
            if not missing_list:
                continue

            # Derive country ucode from filename (e.g. "RUS_V1_flood_admin2.csv" -> "RUS_V1")
            parts = filename.split("_")
            country_ucode = f"{parts[0]}_{parts[1]}"

            print(f"\n  File: {filename} — {len(missing_list)} missing units")
            country_df = None

            for idx, adm2_ucode in enumerate(missing_list):
                now_unit = pd.Timestamp.now()
                print(f"  [{idx+1}/{len(missing_list)}] {adm2_ucode}")

                stats_df = utils.process_missing_adm2_unit(
                    hazard_images=hazard_images,
                    hazard_obj=hazard_obj,
                    country_ucode=country_ucode,
                    adm2_ucode=adm2_ucode,
                    geojson_cache=geojson_cache,
                    properties_cache=properties_cache,
                )

                elapsed = pd.Timestamp.now() - now_unit
                if len(stats_df) > 0:
                    print(f"    Done: {len(stats_df)} rows in {elapsed}")
                    country_df = stats_df if country_df is None else pd.concat([country_df, stats_df])
                else:
                    print(f"    No results in {elapsed}")

            if country_df is None:
                print(f"  Nothing to merge for {filename}")
                continue

            # Merge results into the existing partial file
            existing_path = f"{OUTPUT_FOLDER}/hazards/{hazard_obj.code}/partials/{filename}"
            utils.merge_missing_results(
                new_df=country_df,
                existing_file_path=existing_path,
                output_path=existing_path,  # overwrite in place
            )
            print(f"  Merged {len(country_df)} rows into {existing_path}")

        print(f"Completed {hazard_obj.name} fill in {pd.Timestamp.now() - now_haz}")

    print("\nPhase 2 complete.")
else:
    print("No missing data to fill.")

## Phase 3 — Validate Results

Re-scan the output folder to check for any remaining gaps after Phase 2.

In [ ]:
# ============================================================
# Phase 3: Validate — check for remaining missing ADM2 units
# ============================================================
if RUN_MISSING_DATA_FILL:
    remaining_errors = utils.find_missing_adm2(OUTPUT_FOLDER, all_adm2_ucodes)

    remaining_total = 0
    for haz_code, file_errors in remaining_errors.items():
        missing_count = sum(len(m) for _, m in file_errors)
        if missing_count > 0:
            remaining_total += missing_count
            print(f"  {haz_code}: {missing_count} still missing")
            for fname, missing in file_errors:
                print(f"    {fname}: {missing}")

    if remaining_total == 0:
        print("All ADM2 units are present — no gaps remaining.")
    else:
        print(f"\n{remaining_total} ADM2 units still missing across all hazards.")